# PAVO-Bench · Quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vnmoorthy/pavo-bench/blob/main/notebooks/quickstart.ipynb)

This notebook runs the PAVO-Bench evaluator on Google Colab (free tier, CPU runtime is fine) in ~2 minutes.

You'll:
1. `pip install pavo-bench`
2. Pull 10,000 test turns from HuggingFace
3. Benchmark three reference routers (Always-Cloud, Always-Edge, Hybrid)
4. Load the released 85K-parameter PAVO router and compare
5. Write your own router in five lines and bake it off

The underlying data — latency / quality / cost / energy priors — comes from the committed `tier2_e2e_results.json` and `component_ablation_results.json` files in the repo, so numbers line up with the TMLR paper.

## 1. Install

In [ ]:
# Install from GitHub (PyPI release coming; uses the main branch for now).
!pip install -q git+https://github.com/vnmoorthy/pavo-bench.git

## 2. Load the test split (10,000 turns)

In [ ]:
from pavo_bench import load_dataset
turns = load_dataset(split='test')
print(f'{len(turns):,} turns loaded.')
print('Complexity distribution:', {c: sum(1 for t in turns if t.complexity == c) for c in range(1, 6)})
print('Noise types:', sorted({t.noise_type for t in turns}))
print('Example turn:\n ', turns[0])

## 3. Benchmark three reference routers

In [ ]:
from pavo_bench import (
    AlwaysCloudRouter, AlwaysEdgeRouter, HybridRouter, benchmark_router,
)

for R in [AlwaysCloudRouter(), AlwaysEdgeRouter(), HybridRouter()]:
    r = benchmark_router(R, turns)
    print(f'{r.router:<20s}  P95={r.latency_ms_p95:>7.0f} ms   '
          f'quality={r.quality_mean:.3f}   cost=${r.cost_usd_mean:.3f}   '
          f'energy={r.energy_mj_mean:>6.1f} mJ   infeasible={r.infeasible_pct:.1f}%')

## 4. Load the released PAVO meta-controller

85,041 parameters, trained with multi-objective PPO in 106 s on an A100. You're loading it on CPU here; it's tiny.

In [ ]:
from pavo_bench import PretrainedPAVORouter

# repo_root = the folder where you cloned pavo-bench. Omit this if you set
# the env var PAVO_BENCH_ROOT, or run in the repo root.
!git clone --depth 1 https://github.com/vnmoorthy/pavo-bench.git /tmp/pavo-bench 2>/dev/null || true
pavo = PretrainedPAVORouter.from_released(repo_root='/tmp/pavo-bench')
print(f'PAVO loaded: {pavo.model.count_params():,} params')

r = benchmark_router(pavo, turns)
print(f'{r.router:<20s}  P95={r.latency_ms_p95:>7.0f} ms   '
      f'quality={r.quality_mean:.3f}   cost=${r.cost_usd_mean:.3f}   '
      f'energy={r.energy_mj_mean:>6.1f} mJ   profiles={ {k: round(v,3) for k,v in r.profile_distribution.items()} }')

## 5. Write your own router in five lines

Any callable `PAVOBenchTurn -> {cloud_premium, ondevice_fast, hybrid_balanced}` works. Here's one that uses complexity and network conditions:

In [ ]:
from pavo_bench import BaseRouter

class MyRouter(BaseRouter):
    name = 'Complexity+RTT'
    def route(self, turn):
        # Expensive turns go cloud, but only when network is good.
        if turn.complexity >= 4 and turn.rtt_ms < 150:
            return 'cloud_premium'
        if turn.complexity == 1:
            return 'ondevice_fast'
        return 'hybrid_balanced'

r = benchmark_router(MyRouter(), turns)
print(f'{r.router:<20s}  P95={r.latency_ms_p95:>7.0f} ms   '
      f'quality={r.quality_mean:.3f}   cost=${r.cost_usd_mean:.3f}   '
      f'energy={r.energy_mj_mean:>6.1f} mJ   profiles={ {k: round(v,3) for k,v in r.profile_distribution.items()} }')

## 6. Plot the coupling cliff

The finding from the paper: downstream LLM quality collapses when upstream ASR WER crosses 2%.

In [ ]:
import json, matplotlib.pyplot as plt

with open('/tmp/pavo-bench/tier1_coupling_results.json') as f:
    cc = json.load(f)

wer = sorted(int(k.split('_')[1]) for k in cc['results'])
q = [cc['results'][f'wer_{w}']['mean_quality'] for w in wer]
plt.figure(figsize=(7, 4))
plt.plot(wer, q, marker='o', linewidth=2, color='#d1495b')
plt.axvspan(0, 2, alpha=0.08, color='#2e933c', label='clean regime (WER ≤ 2%)')
plt.axvspan(2, 20, alpha=0.05, color='#d1495b', label='coupling cliff')
plt.xlabel('Injected ASR WER (%)'); plt.ylabel('Mean LLM quality')
plt.title('Coupling cliff (llama3.1:8b)'); plt.grid(alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()

## What's next

- Read the blog post: [`BLOG.md`](https://github.com/vnmoorthy/pavo-bench/blob/main/BLOG.md)
- Run the real pipeline (needs a GPU + ollama): `python experiments/run_all_experiments.py`
- Open a reproduction report if your router beats ours on quality-latency Pareto